# Import libraries

In [16]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load and recreate split

In [2]:
df = pd.read_csv("../data/adult_cleaned.csv")

X = df.drop("income", axis=1)
y = df["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_test.shape, y_test.shape)

(6033, 14) (6033,)


# Building llm sample

In [3]:
sample_size = 50

X_llm = X_test.sample(sample_size, random_state=42).copy()
y_llm = y_test.loc[X_llm.index].copy()

base_llm_df = X_llm.copy()
base_llm_df["actual"] = y_llm
base_llm_df.reset_index(drop=True, inplace=True)

base_llm_df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,actual
0,47,Local-gov,193012,Masters,14,Divorced,Protective-serv,Not-in-family,Black,Male,0,0,50,United-States,1
1,54,Self-emp-not-inc,158948,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,15,United-States,0
2,37,Private,105803,HS-grad,9,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,40,United-States,0
3,40,Local-gov,27444,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Male,0,0,50,United-States,1
4,36,Private,248708,Assoc-acdm,12,Never-married,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,0


# Creating a prompt template

In [4]:
def make_prompt(row):
    return f"""
You are evaluating a structured profile for income classification.

Profile:
- Age: {row['age']}
- Workclass: {row['workclass']}
- Education: {row['education']}
- Education Num: {row['education_num']}
- Marital Status: {row['marital_status']}
- Occupation: {row['occupation']}
- Relationship: {row['relationship']}
- Race: {row['race']}
- Sex: {row['sex']}
- Capital Gain: {row['capital_gain']}
- Capital Loss: {row['capital_loss']}
- Hours per Week: {row['hours_per_week']}
- Native Country: {row['native_country']}

Task:
Predict whether this person's income is:
0 = <=50K
1 = >50K

Return your answer in exactly this format:
Prediction: <0 or 1>
Reason: <short explanation in 1-2 sentences>
""".strip()

In [5]:
#test
print(make_prompt(base_llm_df.iloc[0]))

You are evaluating a structured profile for income classification.

Profile:
- Age: 47
- Workclass: Local-gov
- Education: Masters
- Education Num: 14
- Marital Status: Divorced
- Occupation: Protective-serv
- Relationship: Not-in-family
- Race: Black
- Sex: Male
- Capital Gain: 0
- Capital Loss: 0
- Hours per Week: 50
- Native Country: United-States

Task:
Predict whether this person's income is:
0 = <=50K
1 = >50K

Return your answer in exactly this format:
Prediction: <0 or 1>
Reason: <short explanation in 1-2 sentences>


# Creating a propmt sheet

In [6]:
prompt_df = pd.DataFrame({
    "prompt": base_llm_df.apply(make_prompt, axis=1),
    "llm_prediction": "",
    "llm_reason": ""
})

prompt_df.to_csv("../results/llm_prompts.csv", index=False)
prompt_df.head()

,prompt,llm_prediction,llm_reason
0,You are evaluating a structured profile for in...,,
1,You are evaluating a structured profile for in...,,
2,You are evaluating a structured profile for in...,,
3,You are evaluating a structured profile for in...,,
4,You are evaluating a structured profile for in...,,


# Loading LLM Outputs

In [7]:
filled_llm = pd.read_csv("../results/llm_prompts.csv")
filled_llm.columns = filled_llm.columns.str.strip().str.lower()

print(filled_llm.columns)
filled_llm.head()

Index(['prompt', 'llm_prediction', 'llm_reason'], dtype='str')


,prompt,llm_prediction,llm_reason
0,You are evaluating a structured profile for in...,0.0,Despite a master’s degree and full-time work p...
1,You are evaluating a structured profile for in...,0.0,Very low weekly hours (15) strongly indicate p...
2,You are evaluating a structured profile for in...,0.0,HS education and a lower-skill occupation (han...
3,You are evaluating a structured profile for in...,1.0,Bachelor’s degree with a professional specialt...
4,You are evaluating a structured profile for in...,0.0,Mid-level education and a craft job with stand...


# Merging back with the ground truth

In [8]:
base_llm_df["llm_prediction"] = filled_llm["llm_prediction"]
base_llm_df["llm_reason"] = filled_llm["llm_reason"]

llm_df = base_llm_df.copy()
llm_df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,actual,llm_prediction,llm_reason
0,47,Local-gov,193012,Masters,14,Divorced,Protective-serv,Not-in-family,Black,Male,0,0,50,United-States,1,0.0,Despite a master’s degree and full-time work p...
1,54,Self-emp-not-inc,158948,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,15,United-States,0,0.0,Very low weekly hours (15) strongly indicate p...
2,37,Private,105803,HS-grad,9,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,40,United-States,0,0.0,HS education and a lower-skill occupation (han...
3,40,Local-gov,27444,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Male,0,0,50,United-States,1,1.0,Bachelor’s degree with a professional specialt...
4,36,Private,248708,Assoc-acdm,12,Never-married,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,0,0.0,Mid-level education and a craft job with stand...


# Keeping only the complete rows

In [9]:
llm_df = llm_df.dropna(subset=["llm_prediction"]).copy()
llm_df["llm_prediction"] = llm_df["llm_prediction"].astype(int)

print("Rows used:", len(llm_df))
llm_df.head()

Rows used: 48


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,actual,llm_prediction,llm_reason
0,47,Local-gov,193012,Masters,14,Divorced,Protective-serv,Not-in-family,Black,Male,0,0,50,United-States,1,0,Despite a master’s degree and full-time work p...
1,54,Self-emp-not-inc,158948,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,15,United-States,0,0,Very low weekly hours (15) strongly indicate p...
2,37,Private,105803,HS-grad,9,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,40,United-States,0,0,HS education and a lower-skill occupation (han...
3,40,Local-gov,27444,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Male,0,0,50,United-States,1,1,Bachelor’s degree with a professional specialt...
4,36,Private,248708,Assoc-acdm,12,Never-married,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,0,0,Mid-level education and a craft job with stand...


# LLM Performance Evaluation

In [10]:
print("LLM Accuracy:", accuracy_score(llm_df["actual"], llm_df["llm_prediction"]))
print("LLM Precision:", precision_score(llm_df["actual"], llm_df["llm_prediction"]))
print("LLM Recall:", recall_score(llm_df["actual"], llm_df["llm_prediction"]))
print("LLM F1:", f1_score(llm_df["actual"], llm_df["llm_prediction"]))

LLM Accuracy: 0.6458333333333334
LLM Precision: 0.4
LLM Recall: 0.42857142857142855
LLM F1: 0.41379310344827586


In [11]:
# results table
llm_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1"],
    "LLM": [
        accuracy_score(llm_df["actual"], llm_df["llm_prediction"]),
        precision_score(llm_df["actual"], llm_df["llm_prediction"]),
        recall_score(llm_df["actual"], llm_df["llm_prediction"]),
        f1_score(llm_df["actual"], llm_df["llm_prediction"])
    ]
})

llm_metrics

,Metric,LLM
0,Accuracy,0.645833
1,Precision,0.400000
2,Recall,0.428571
3,F1,0.413793


# Analyse bias by sex and race

In [12]:
llm_gender = llm_df.groupby("sex").apply(
    lambda x: pd.Series({
        "accuracy": (x["actual"] == x["llm_prediction"]).mean(),
        "positive_rate": x["llm_prediction"].mean(),
        "count": len(x)
    })
)

llm_gender

,accuracy,positive_rate,count
sex,,,
Female,0.800000,0.333333,15.0
Male,0.575758,0.303030,33.0


In [13]:
llm_race = llm_df.groupby("race").apply(
    lambda x: pd.Series({
        "accuracy": (x["actual"] == x["llm_prediction"]).mean(),
        "positive_rate": x["llm_prediction"].mean(),
        "count": len(x)
    })
)

llm_race

,accuracy,positive_rate,count
race,,,
Amer-Indian-Eskimo,1.000000,0.000000,1.0
Asian-Pac-Islander,0.500000,0.500000,2.0
Black,0.500000,0.166667,6.0
White,0.666667,0.333333,39.0


# ML vs LLM Comparison

In [14]:
rf_pipeline = joblib.load("../results/random_forest_pipeline.pkl")
X_test_saved = joblib.load("../results/X_test.pkl")
y_test_saved = joblib.load("../results/y_test.pkl")

comparison = pd.DataFrame({
    "Model": ["Random Forest", "LLM"],
    "Accuracy": [
        accuracy_score(y_test_saved, rf_pipeline.predict(X_test_saved)),
        accuracy_score(llm_df["actual"], llm_df["llm_prediction"])
    ]
})

comparison

,Model,Accuracy
0,Random Forest,0.849494
1,LLM,0.645833


# saving all the results

In [15]:
llm_df.to_csv("../results/llm_predictions_final.csv", index=False)
llm_gender.to_csv("../results/llm_gender_bias.csv")
llm_race.to_csv("../results/llm_race_bias.csv")
comparison.to_csv("../results/ml_vs_llm_comparison.csv", index=False)
llm_metrics.to_csv("../results/llm_metrics.csv", index=False)

## Interpretation

The LLM-based decision system was evaluated on a sampled subset of the test data and compared with the Random Forest model.

The results indicate that while the LLM is capable of generating human-readable reasoning, its predictive performance differs from the traditional machine learning model. In addition, variations in accuracy and positive prediction rates across demographic groups suggest that LLMs may exhibit different bias patterns from structured machine learning systems.

This comparison highlights an important distinction: traditional ML models provide structured, measurable explainability through methods such as SHAP, whereas LLMs generate natural-language justifications that may not always align consistently with predictive behaviour.

These findings support the need for further research into the reliability, fairness, and interpretability of LLM-based decision systems in high-stakes applications.